# Mayel: revisión reproducible de evidencia comercial
El archivo JSON adjunto contiene lecturas acotadas de staging, con referencias y fechas. Este notebook no consulta APIs ni escribe en Seller OS. Los intervalos ilustran incertidumbre; no introducen umbrales globales de tratamiento.

Una instantánea repetida de la misma ventana no añade observaciones independientes. El agregado de toda la tienda es descriptivo: la heterogeneidad de categorías impide declararlo automáticamente un benchmark comparable.

In [1]:
import json, math, statistics
from pathlib import Path
p = Path("assistant-revenue-evidence-blockers-fast-closeout-v1-readback.json")
evidence = json.loads(p.read_text())
candidates = evidence["candidates"]
assert len(candidates) == len({c["itemId"] for c in candidates}) == 23
print("Listings LIVE únicos:", len(candidates))
print("Fechas de lectura:", evidence["recordedAt"])


Listings LIVE únicos: 23
Fechas de lectura: 2026-09-09T17:15:07.130905+00:00


In [2]:
snapshots = [c["latestMetrics"] for c in candidates]
windows = sorted({(s["window_start"], s["window_end"]) for s in snapshots})
views = [s["views"] for s in snapshots if s["views"] is not None]
transactions = [s["transactions"] for s in snapshots if s["transactions"] is not None]
print("Ventanas observadas:", windows)
print("Views totales / máximo individual:", sum(views), max(views))
print("Transactions observadas:", sum(transactions))
print("Mediana descriptiva de impresiones:", statistics.median(s["impressions"] for s in snapshots))
assert sum(views) == 82 and max(views) == 14 and sum(transactions) == 0


Ventanas observadas: [('2026-08-10', '2026-09-08')]
Views totales / máximo individual: 82 14
Transactions observadas: 0
Mediana descriptiva de impresiones: 184


In [3]:
selected = next(c for c in candidates if c["itemId"] == evidence["selectedCandidate"])
x, n = selected["source"]["calculatedCtrNumerator"], selected["source"]["calculatedCtrDenominator"]
z = 1.959963984540054
p_hat = x/n
den = 1+z*z/n
center = (p_hat+z*z/(2*n))/den
half = z*math.sqrt(p_hat*(1-p_hat)/n+z*z/(4*n*n))/den
print("CTR fuente (numerador/denominador):", x, n, p_hat)
print("Wilson 95%, exploratorio:", (center-half, center+half))
assert selected["latestMetrics"]["transactions"] == 0
nv = selected["latestMetrics"]["views"]
print("Cero ventas; límite superior exacto bilateral 95%:", 1-0.025**(1/nv))
print("Tratamiento conservado: TEST; sin conclusión fuerte sobre conversión.")


CTR fuente (numerador/denominador): 2 265 0.007547169811320755
Wilson 95%, exploratorio: (0.0020721520357035363, 0.02709544307812914)
Cero ventas; límite superior exacto bilateral 95%: 0.7075982261787134
Tratamiento conservado: TEST; sin conclusión fuerte sobre conversión.


## Economía y autoridad
La confirmación OWNER permite cero otros costes variables únicamente cuando producto, Shipping y comisiones están demostrados. Un coste desconocido no es cero. Los dos registros Shipping previos y las dos representaciones de presencia LIVE se distinguen en el JSON: no se suman ni se intercambian como si fueran la misma autoridad.

Fuentes oficiales: [fees de eBay](https://www.ebay.com/help/selling/fees-credits-invoices/selling-fees?id=4822) y [Finances API](https://developer.ebay.com/develop/api/sell/finances_api). Su documentación no constituye por sí sola prueba del importe aplicable a este listing.

In [4]:
print("Política OWNER:", evidence["ownerCostPolicy"])
print("Resultado de cierre:", evidence["fastCloseout"]["status"])
print("Blockers:", evidence["blockers"])
assert evidence["canaryExecution"]["marketplaceWrites"] == 0
assert evidence["canaryExecution"]["ebayAdsWrites"] == 0


Política OWNER: {'contractVersion': 'SELLER_OS_OWNER_VARIABLE_COST_POLICY_V1', 'recordedAt': '2026-09-09T16:49:22.982857+00:00', 'confirmationSource': 'EXPLICIT_OWNER_MESSAGE_IN_THIS_SESSION_2026_09_09', 'marketplaceAccountKey': 'imnova-ebay-us-primary:cd8fd3dc2b4102d4aff320268c647fa895c6416df01013f9bb06b3a587709e12', 'marketplaceId': 'EBAY_US', 'environment': 'DEDICATED_PREPROD_ONLY', 'OWNER_VARIABLE_COST_POLICY_CONFIRMED': True, 'OTHER_VARIABLE_COSTS_POLICY': 'NONE_CURRENTLY_APPLICABLE', 'OTHER_PROVEN_VARIABLE_COSTS': 0, 'currency': 'USD', 'ZERO_IS_EXPLICIT_POLICY_NOT_ASSUMPTION': True, 'scope': 'CURRENTLY_OPERATED_SELLER_OS_LISTINGS', 'observedCurrentItemIds': ['366574069492', '366581718546', '366582544476', '366582586826', '366582671136', '366592485792', '366592919965', '366597434810', '366634810965', '366635285436', '366643122092', '366643126310', '366643190059', '366643555454', '366647547173', '366649437609', '366649508886', '366650047727', '366650054490', '366650065203', '366650